In [24]:
!pip install chromadb sentence-transformers -q

In [25]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb
print("All libraries imported successfully")
print(f"ChromaDB version: {chromadb.__version__}")

All libraries imported successfully
ChromaDB version: 1.5.9


In [26]:
documents=[
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobiles",
    "SQL is used to query database",
    "Machine learning trains models on data",
]
query_keyword='vehicle'
print('='*60)
print(f'KEYWORD SEARCHED for: {query_keyword}')
print("="*60)
for i,doc in enumerate(documents):
  if query_keyword.lower() in doc.lower():
    print(f"  FOUND  [doc_{i}]: {doc}")
  else:
    print(f"  MISSED [doc_{i}]:{doc}")
print()
print("PROBLEM: doc_2 talks about 'Cars and trucks' - which ARE vehicles!")
print("But keyword search MISSED it because it searched for the text exact word 'vehicle'.")

KEYWORD SEARCHED for: vehicle
  MISSED [doc_0]:ETL is used to clean and transform data
  FOUND  [doc_1]: A vehicle is a mode of transportation
  MISSED [doc_2]:Cars and trucks are popular automobiles
  MISSED [doc_3]:SQL is used to query database
  MISSED [doc_4]:Machine learning trains models on data

PROBLEM: doc_2 talks about 'Cars and trucks' - which ARE vehicles!
But keyword search MISSED it because it searched for the text exact word 'vehicle'.


In [27]:
failure_examples = [
    {"query":"I feel sick",  "misses":"Iam unwell, patient has fever"},
    {"query":"How to cook rice",  "misses":"Steps to prepare rice"},
    {"query":"vehicle speed",  "misses":"car acceleration,automobile velocity"},
    {"query":"ML model accuracy", "misses":"classification performance, prediction quality"},
]
print("KEYWORD SEARCH FAILURE CASES")
print("="*60)
for ex in failure_examples:
  print(f"Query: '{ex['query']}'")
  print(f"Misses: '{ex['misses']}'")
  print("-"*40)
print()
print("SOLUTION: We need search that understands MEANING, not just characters.")
print("That is what EMBEDDINGS do.")

KEYWORD SEARCH FAILURE CASES
Query: 'I feel sick'
Misses: 'Iam unwell, patient has fever'
----------------------------------------
Query: 'How to cook rice'
Misses: 'Steps to prepare rice'
----------------------------------------
Query: 'vehicle speed'
Misses: 'car acceleration,automobile velocity'
----------------------------------------
Query: 'ML model accuracy'
Misses: 'classification performance, prediction quality'
----------------------------------------

SOLUTION: We need search that understands MEANING, not just characters.
That is what EMBEDDINGS do.


In [28]:
print("Loading embedding model... (may take 1-2 minutes on forst run)")
model =SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded successfully")
print(f"Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")
print()
sentence = "ETL is used to clean and transform data"
embedding=model.encode(sentence)
print(f"Input sentence: '{sentence}'")
print()
print(f"Embedding type:{type(embedding)}")
print(f"Embedding shape:{embedding.shape}")
print(f"First 10 numbers: {embedding[:10].round(4)}")
print()
print(f"Min value: {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")

Loading embedding model... (may take 1-2 minutes on forst run)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully
Model produces vectors of size: 384 dimensions

Input sentence: 'ETL is used to clean and transform data'

Embedding type:<class 'numpy.ndarray'>
Embedding shape:(384,)
First 10 numbers: [-0.0784  0.0541  0.0224 -0.0389  0.0221 -0.0904  0.0007 -0.0152  0.0733
  0.0362]

Min value: -0.1381
Max value: 0.1815


/tmp/ipykernel_2174/2994884856.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")


In [29]:
sentences=[
    "ETL is used to clean and transform data",
    "Data tranformation is a key pipeline step",
    "The sky is blue and clouds are white",
]
embeddings=model.encode(sentences)
print(f"Number of sentences: {len(sentences)}")
print(f"Shape of embeddings array: {embeddings.shape}")
print()
print("Each row is one sentence's embedding:")
for i,sent in enumerate(sentences):
  print(f"  Sentence {i}: shape={embeddings[i].shape}, first 5 values={embeddings[i][:5].round(3)}")

Number of sentences: 3
Shape of embeddings array: (3, 384)

Each row is one sentence's embedding:
  Sentence 0: shape=(384,), first 5 values=[-0.078  0.054  0.022 -0.039  0.022]
  Sentence 1: shape=(384,), first 5 values=[-0.052  0.029 -0.004 -0.02  -0.063]
  Sentence 2: shape=(384,), first 5 values=[0.054 0.06  0.075 0.049 0.05 ]


In [30]:
def cosine_similarity(vec_a,vec_b):
  dot_product=np.dot(vec_a,vec_b)
  norm_a=np.linalg.norm(vec_a)
  norm_b=np.linalg.norm(vec_b)
  return dot_product/(norm_a*norm_b)
sim_01 = cosine_similarity(embeddings[0],embeddings[1])
sim_02 = cosine_similarity(embeddings[0],embeddings[2])
sim_03 = cosine_similarity(embeddings[1],embeddings[2])

print("COSINE SIMILARITY SCORES")
print("="*60)
print(f"Sentence 0: {sentences[0]}")
print(f"Sentence 1: {sentences[1]}")
print(f"Sentence 2: {sentences[2]}")
print()
print(f"Similarity (0 vs 1):{sim_01:.4f} <- Expected: HIGH (same topic)")
print(f"Similarity (0 vs 2):{sim_01:.4f} <- Expected: LOW (different topic)")
print(f"Similarity (1 vs 2):{sim_01:.4f} <- Expected: LOW (different topic)")
print()
print("INSIGHT: Sentences 0 and 1 have different words but similar meaning.")
print("Their cosne similarity score is high - the embedding captured the meaning")

COSINE SIMILARITY SCORES
Sentence 0: ETL is used to clean and transform data
Sentence 1: Data tranformation is a key pipeline step
Sentence 2: The sky is blue and clouds are white

Similarity (0 vs 1):0.4660 <- Expected: HIGH (same topic)
Similarity (0 vs 2):0.4660 <- Expected: LOW (different topic)
Similarity (1 vs 2):0.4660 <- Expected: LOW (different topic)

INSIGHT: Sentences 0 and 1 have different words but similar meaning.
Their cosne similarity score is high - the embedding captured the meaning


In [31]:
your_sentences=[
    "Machine learning trains model on labeled data",
    "AI algorithms learn patterns from example",
    "I enjoy eating pizza for lunch"
]
your_embeddings=model.encode(your_sentences)
sim_your_01=cosine_similarity(your_embeddings[0],your_embeddings[1])
sim_your_02=cosine_similarity(your_embeddings[0],your_embeddings[2])

print("COSINE SIMILARITY SCORES")
print("="*60)
print(f"Sentence 0: {your_sentences[0]}")
print(f"Sentence 1: {your_sentences[1]}")
print(f"Sentence 2: {your_sentences[2]}")
print()
print(f"Similarity (A vs B):{sim_your_01:.4f}")
print(f"Similarity (A vs B):{sim_your_01:.4f}")
print()
if sim_your_01 > sim_your_02:
  print("Correct: A and B are more similar to each other that A and C")
else:
  print("Incorrect: A and C are more similar to each other that A and B")

COSINE SIMILARITY SCORES
Sentence 0: Machine learning trains model on labeled data
Sentence 1: AI algorithms learn patterns from example
Sentence 2: I enjoy eating pizza for lunch

Similarity (A vs B):0.3139
Similarity (A vs B):0.3139

Correct: A and B are more similar to each other that A and C


# What is ChromaDB

chromaDB is a vector database designed to store embeddings and retrieve the most similar ones quickly.

**chromaDB Key Terms:**


*   Collection  - Like a table in SQL
*   Document    - The actual text content you store
*   MetaData    - Extra information about the document
*   ID          - Unique identifier for each document
*   DataFrame

**CRITICAL RULE:** Distance vs Similarity
chromadb returns distance not similarity distances



*   Distance 0.0 = Perfect match
*   Distance 1.0 = no match
*   This is the Opposite of cosine similarity score direction

In [32]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection("demo_notes")
print("ChromaDB client created (in-memory mode)")
print(f"Collection name: demo_notes")
print(f"Documents in collection: {collection.count()}")

ChromaDB client created (in-memory mode)
Collection name: demo_notes
Documents in collection: 5


In [33]:
sample_docs=[
    "ETL stands for Extract Transform Load - the core data engineering process",
    "SQL SELECT statements retrieve data from database tables",
    "Machine learning models learn patterns from training data",
    "Python Pandas library is used for data manipulation and cleaning",
    "Neural networks are inspired by how the human brain works",
]
sample_ids=['doc001','doc002','doc003','doc004','doc005']
sample_metadata=[
    {"subject":"Data Engineering","topic":"ETL"},
    {"subject":"Data Engineering","topic":"SQL"},
    {"subject":"Machine Learning","topic":"ML Basics"},
    {"subject":"Python","topic":"Pandas"},
    {"subject":"Machine Learning","topic":"Neural Networks"},
]
collection.add(
    documents=sample_docs,
    ids=sample_ids,
    metadatas=sample_metadata
)
print(f"Documents added to collection!")
print(f"Total documents now in collection: {collection.count()}")

Documents added to collection!
Total documents now in collection: 5


In [34]:
query ="How do I clean and prepare data"
results = collection.query(
    query_texts=query, n_results = 2
)
print("RESULT KEY AVAILABLE:")
print(list(results.keys()))

RESULT KEY AVAILABLE:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [35]:
print(f"Query:'{query}")
print("="*50)
print()
matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distances = results['distances'][0]
matched_metadata = results['metadatas'][0]

Query:'How do I clean and prepare data



In [36]:
print(f"Query:'{query}")
print("="*50)
print()
matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distances = results['distances'][0]
matched_metadata = results['metadatas'][0]

Query:'How do I clean and prepare data



In [37]:
filtered_query="How to computers learn from examples?"
filtered_result=collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"Subject":"Machine Learning"},
    #where_document={"$contains":"neural networks"}
)
print(f"Filtered query: '{filtered_query}'")
print("Filter:only machine leaning documents")
print("="*60)
for rank,(doc,dist,distance,metadata) in enumerate(zip(
    filtered_result['documents'][0],
    filtered_result['distances'][0],
    filtered_result['metadatas'][0]
),start=1):
  print(f"Rank {rank}|Distance{dist:4f}|subject:{meta['subject']}")
  print(f"  {doc}")
  print()
print("Notice:Only Ml ")


Filtered query: 'How to computers learn from examples?'
Filter:only machine leaning documents
Notice:Only Ml 


In [38]:
print("DISTANCE TO SIMILARITY CONVERSION")
print("="*60)
print(f"{'Distance':<15} {'Similarity':<15}{'Interpretation':<20}")
print("="*60)
diatance=[0.05,0.20,0.40,0.65,0.90]
interpretation=["Near identical","Very similar","Related","Somewhat related","Not Related"]
for dist,interp in zip(diatance,interpretation):
  similarity=1-dist
  print(f"{dist:<15.2f} {similarity:<15.2f} {interp:<20}")

DISTANCE TO SIMILARITY CONVERSION
Distance        Similarity     Interpretation      
0.05            0.95            Near identical      
0.20            0.80            Very similar        
0.40            0.60            Related             
0.65            0.35            Somewhat related    
0.90            0.10            Not Related         


In [39]:
notes_df=pd.read_csv('college_notes.csv')
print("Dataset loaded")
print(f"Shape of dataset: {notes_df.shape}")
print(f"Columns in dataset: {list(notes_df.columns)}")

Dataset loaded
Shape of dataset: (15, 4)
Columns in dataset: ['note_id', 'subject', 'topic', 'content']


In [40]:
print("Notes per subject:")
print(notes_df['subject'].value_counts())
first_note=notes_df.iloc[0]

print(f"Note ID:{first_note['note_id']}")
print(f"Subject:{first_note['subject']}")
print(f"Content:{first_note['content']}")
print(f"Topic:{first_note['topic']}")


Notes per subject:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64
Note ID:N001
Subject:Data Engineering
Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
Topic:ETL Pipelines


In [41]:
all_documents=notes_df['content'].tolist()
all_ids=notes_df['note_id'].tolist()
all_metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for _,row in notes_df.iterrows()
]
print(f"Total documents: {len(all_documents)}")
print(f"ID prepared: {all_ids[:5]}")
print(f"Metadata prepared: {all_metadatas[:5]}")
print()
print("Sample ID:")

Total documents: 15
ID prepared: ['N001', 'N002', 'N003', 'N004', 'N005']
Metadata prepared: [{'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}, {'subject': 'Data Engineering', 'topic': 'SQL Databases'}, {'subject': 'Data Engineering', 'topic': 'Data Cleaning'}, {'subject': 'Data Engineering', 'topic': 'APIs and Data Collection'}, {'subject': 'Data Engineering', 'topic': 'Big Data and PySpark'}]

Sample ID:


#MINI PROJECT
##SMART NOTES SEARCH ENGINE


In [42]:
client = chromadb.Client()

collection = client.create_collection(
    name="college_notes"
)

In [43]:

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:

documents = notes_df["content"].tolist()

embeddings = model.encode(
    documents
).tolist()

In [48]:
collection.add(
    ids=notes_df["note_id"].astype(str).tolist(),
    documents=notes_df["content"].tolist(),
    embeddings=embeddings,
    metadatas=[
        {
            "subject": row["subject"],
            "topic": row["topic"]
        }
        for _, row in notes_df.iterrows()
    ]
)

print("\nAll Notes Indexed Successfully!")


All Notes Indexed Successfully!


In [49]:
query = "How is data extracted and loaded?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("\n" + "="*60)
print("SEMANTIC SEARCH RESULTS")
print("="*60)

for i, doc in enumerate(results["documents"][0], start=1):
    print(f"\nResult {i}")
    print(doc)


SEMANTIC SEARCH RESULTS

Result 1
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

Result 2
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.

Result 3
Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.


In [50]:
query = "database queries"

query_embedding = model.encode(query).tolist()

filtered_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"subject": "Data Engineering"}
)

print("\n" + "="*60)
print("FILTERED RESULTS")
print("="*60)

for i, doc in enumerate(filtered_results["documents"][0], start=1):
    print(f"\nResult {i}")
    print(doc)


FILTERED RESULTS

Result 1
A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.

Result 2
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.

Result 3
Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.


In [51]:

keyword = "database"

keyword_results = notes_df[
    notes_df["content"].str.contains(
        keyword,
        case=False,
        na=False
    )
]

print("\n" + "="*60)
print("KEYWORD SEARCH RESULTS")
print("="*60)

for _, row in keyword_results.iterrows():
    print("\nTopic:", row["topic"])
    print(row["content"])



KEYWORD SEARCH RESULTS

Topic: ETL Pipelines
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

Topic: SQL Databases
A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
